In [ ]:
!sudo apt-get update
!sudo apt-get install cuda-12-4

In [ ]:
!rm /usr/local/cuda
!ln -s /usr/local/cuda-12.4 /usr/local/cuda

In [ ]:
!nvcc --version
!pip install nvcc4jupyter

In [ ]:
%load_ext nvcc4jupyter

In [ ]:
!apt-get -y install postgresql postgresql-contrib
!service postgresql start
!sudo -u postgres psql -c "ALTER USER postgres PASSWORD 'postgres';"
!sudo -u postgres createdb testdb
!pip install psycopg2-binary

In [ ]:
# 🗃️ Create tables and insert 1M rows of data
import psycopg2
import numpy as np
import time

conn = psycopg2.connect(
    dbname="testdb",
    user="colabuser",
    password="colabpass",
    host="localhost"
)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS A, B")
cursor.execute("CREATE TABLE A(id INT, value_a INT)")
cursor.execute("CREATE TABLE B(id INT, value_b INT)")
conn.commit()

# Generate data (1M each)
A_data = [(i, np.random.randint(1, 10000)) for i in range(1, 1000001)]
B_data = [(i, np.random.randint(1, 10000)) for i in range(500000, 1500000)]

# Insert using batched queries
import psycopg2.extras as extras

def batch_insert(cursor, query, data):
    extras.execute_batch(cursor, query, data, page_size=10000)

print("Inserting into A...")
batch_insert(cursor, "INSERT INTO A VALUES (%s, %s)", A_data)
print("Inserting into B...")
batch_insert(cursor, "INSERT INTO B VALUES (%s, %s)", B_data)
conn.commit()
